## Question 1

In [ ]:
%pip install -q nltk scikit-learn pandas matplotlib seaborn

In [ ]:
import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

print("Setup complete")

In [ ]:
corpus = {
    "glucose_monitoring": [
        "How do I check my blood sugar at home",
        "When should I test my glucose levels",
        "What is a normal fasting blood glucose reading",
        "How often should a person with diabetes monitor sugar",
        "How do I use a glucometer correctly",
        "Should I wash my hands before checking blood sugar",
        "What does a post meal glucose reading mean",
        "How can I record daily blood glucose readings"
    ],
    "hba1c": [
        "What does the HbA1c test measure",
        "How often is an A1c test usually done",
        "Explain the meaning of glycated hemoglobin",
        "What is the difference between A1c and daily glucose",
        "Why does my doctor order an HbA1c test",
        "How does A1c show average blood sugar",
        "What information does an HbA1c report provide",
        "Can I understand long term glucose control using A1c"
    ],
    "insulin_use": [
        "What is insulin used for in diabetes",
        "How should insulin be stored according to its label",
        "Why do some people with diabetes need insulin",
        "What are the common types of insulin",
        "Where can I read the instructions for an insulin pen",
        "What should I know before using an insulin device",
        "Why is insulin timing important",
        "How do I dispose of used insulin needles safely"
    ],
    "medication_safety": [
        "What should I do if I miss a diabetes medicine dose",
        "Can diabetes medicines cause side effects",
        "Why should I follow the prescription label",
        "Should I change my medicine dose myself",
        "How can I keep track of my diabetes medicines",
        "What should I ask my pharmacist about a new medicine",
        "Can I stop taking prescribed diabetes medicine",
        "How should I store tablets safely"
    ],
    "healthy_diet": [
        "What foods are suitable for a balanced diabetes diet",
        "Why is portion control important for diabetes",
        "Which foods contain dietary fiber",
        "How can I reduce added sugar in meals",
        "What is the role of carbohydrates in blood glucose",
        "Are whole grains useful in a balanced diet",
        "How can I plan balanced meals",
        "Why should I read nutrition labels"
    ],
    "physical_activity": [
        "What are the benefits of walking for people with diabetes",
        "How can physical activity support glucose management",
        "Why should I check with my doctor before a new exercise plan",
        "What are examples of moderate physical activity",
        "How can I build a regular walking routine",
        "Why is warming up before exercise useful",
        "What should I consider when exercising with diabetes",
        "How does regular activity support general health"
    ],
    "diabetes_symptoms": [
        "What are common symptoms of diabetes",
        "Why can diabetes cause frequent urination",
        "Can increased thirst be a sign of high blood sugar",
        "What does blurred vision mean in diabetes",
        "Why might someone with diabetes feel unusually tired",
        "What are possible signs of low blood sugar",
        "Can slow healing wounds be associated with diabetes",
        "What symptoms should I discuss with my healthcare provider"
    ],
    "emergency": [
        "I am confused and sweating and my blood sugar is very low",
        "Someone with diabetes is unconscious what should I do",
        "I have severe difficulty breathing and feel faint",
        "A person is having a seizure after a glucose problem",
        "I have severe vomiting and deep rapid breathing",
        "The person is unresponsive and cannot swallow",
        "I think this is a life threatening diabetes emergency",
        "There is severe confusion and loss of consciousness"
    ]
}

rows = [
    {"query": query, "intent": intent}
    for intent, queries in corpus.items()
    for query in queries
]
data = pd.DataFrame(rows)

assert len(data) >= 40
assert data["intent"].nunique() >= 5
assert all(data.groupby("intent").size() >= 2)

print("Total queries:", len(data))
print("Intent categories:", data["intent"].nunique())
display(data.groupby("intent").size().to_frame("query_count"))
display(data)

In [ ]:
taxonomy = {
    "Monitoring": {
        "Glucose Monitoring": "glucose_monitoring",
        "HbA1c Testing": "hba1c"
    },
    "Medication": {
        "Insulin Use": "insulin_use",
        "Medication Safety": "medication_safety"
    },
    "Lifestyle": {
        "Healthy Diet": "healthy_diet",
        "Physical Activity": "physical_activity"
    },
    "Symptoms and Safety": {
        "Diabetes Symptoms": "diabetes_symptoms",
        "Emergency Support": "emergency"
    }
}

for major, subcategories in taxonomy.items():
    print(major)
    for subcategory, intent in subcategories.items():
        print("  ", subcategory, "->", intent)

In [ ]:
domain_concepts = [
    "Diabetes", "Blood Glucose", "Glucometer", "HbA1c",
    "Insulin", "Medication", "Nutrition", "Carbohydrates",
    "Physical Activity", "Hypoglycemia", "Hyperglycemia",
    "Healthcare Provider"
]

relationships = [
    ("Diabetes", "is monitored using", "Blood Glucose"),
    ("Blood Glucose", "is measured by", "Glucometer"),
    ("HbA1c", "estimates", "Long-term Blood Glucose"),
    ("Insulin", "helps regulate", "Blood Glucose"),
    ("Medication", "must follow", "Prescription Instructions"),
    ("Nutrition", "influences", "Blood Glucose"),
    ("Carbohydrates", "are part of", "Nutrition"),
    ("Physical Activity", "supports management of", "Diabetes"),
    ("Hypoglycemia", "means low", "Blood Glucose"),
    ("Hyperglycemia", "means high", "Blood Glucose"),
    ("Healthcare Provider", "interprets", "HbA1c"),
    ("Healthcare Provider", "guides", "Medication Use")
]

assert len(domain_concepts) >= 8
assert len(relationships) >= 10

print("Domain concepts:", len(domain_concepts))
print("Relationships:", len(relationships))
display(pd.DataFrame(relationships, columns=["Concept 1", "Relationship", "Concept 2"]))

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and token not in string.punctuation
    ]
    return " ".join(tokens)

data["processed_query"] = data["query"].apply(preprocess_text)
display(data[["query", "processed_query", "intent"]].head(12))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["processed_query"],
    data["intent"],
    test_size=0.25,
    random_state=42,
    stratify=data["intent"]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, average="weighted", zero_division=0)
recall = recall_score(y_test, predictions, average="weighted", zero_division=0)
f1 = f1_score(y_test, predictions, average="weighted", zero_division=0)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))
print(classification_report(y_test, predictions, zero_division=0))

In [ ]:
labels = sorted(data["intent"].unique())
matrix = confusion_matrix(y_test, predictions, labels=labels)

plt.figure(figsize=(12, 8))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels
)
plt.xlabel("Predicted intent")
plt.ylabel("Actual intent")
plt.title("Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
knowledge_base = {
    "glucose_monitoring": "Blood glucose monitoring helps a person understand glucose patterns. Follow the monitoring schedule and technique recommended by a healthcare professional. Record readings with their dates, times and relevant context.",
    "hba1c": "HbA1c is a blood test that reflects average blood glucose over approximately the previous two to three months. A healthcare professional interprets the result in the context of the person's health.",
    "insulin_use": "Insulin is a hormone used as a medicine by some people with diabetes. Insulin type, timing, dose and storage depend on the prescribed product and individual treatment plan. Follow the product instructions and clinician's advice.",
    "medication_safety": "Take diabetes medicines according to the prescription and product instructions. Do not start, stop or change a dose based on an assistant response. Ask a doctor or pharmacist about missed doses, interactions, storage and side effects.",
    "healthy_diet": "A balanced eating pattern can include vegetables, whole grains, fiber-rich foods, suitable protein sources and appropriate portions. Nutrition needs vary; a registered dietitian or healthcare professional can help create an individual plan.",
    "physical_activity": "Regular physical activity can support general health and diabetes management. Choose activities appropriate to ability and health status. Discuss a new exercise plan with a healthcare professional, especially when complications or other conditions are present.",
    "diabetes_symptoms": "Possible diabetes symptoms include increased thirst, frequent urination, increased hunger, fatigue, blurred vision and slow-healing wounds. Symptoms can have different causes and should be discussed with a healthcare professional.",
    "emergency": "Severe confusion, unconsciousness, seizures, inability to swallow, severe breathing difficulty or other life-threatening symptoms require immediate emergency medical help. Contact local emergency services. Do not give food or drink to an unconscious person."
}

assert set(knowledge_base) == set(data["intent"].unique())
print("Knowledge base categories:", len(knowledge_base))

In [ ]:
emergency_patterns = [
    r"unconscious", r"unresponsive", r"cannot swallow", r"can't swallow",
    r"seizure", r"severe breathing", r"trouble breathing",
    r"life threatening", r"loss of consciousness",
    r"severe confusion", r"deep rapid breathing"
]

domain_terms = {
    "diabetes", "diabetic", "glucose", "blood sugar", "sugar level",
    "hba1c", "a1c", "insulin", "glucometer", "hypoglycemia",
    "hyperglycemia", "carbohydrate", "diabetes medicine", "diabetes medication"
}

def is_emergency(query):
    return any(re.search(pattern, query.lower()) for pattern in emergency_patterns)

def is_supported_domain(query):
    q = query.lower()
    return any(term in q for term in domain_terms)

def retrieve_response(query, confidence_threshold=0.30, margin_threshold=0.05):
    query = str(query).strip()

    if not query:
        return "Please enter a healthcare query."

    if is_emergency(query):
        return (
            "EMERGENCY ALERT: This may be a medical emergency. "
            "Contact local emergency services immediately or go to the nearest emergency department. "
            "Do not rely on this assistant for emergency care."
        )

    if not is_supported_domain(query):
        return (
            "Unsupported query: I am designed to provide general diabetes-care information. "
            "Please consult a qualified healthcare professional for other medical topics."
        )

    processed = preprocess_text(query)
    probabilities = model.predict_proba([processed])[0]
    order = np.argsort(probabilities)[::-1]
    best_index = order[0]
    confidence = float(probabilities[best_index])
    second_confidence = float(probabilities[order[1]]) if len(order) > 1 else 0.0
    intent = model.classes_[best_index]

    if confidence < confidence_threshold or confidence - second_confidence < margin_threshold:
        return (
            "Low confidence: I could not identify your diabetes-care question reliably. "
            "Please rephrase it or consult a healthcare professional."
        )

    return (
        f"Detected intent: {intent}\n"
        f"Confidence: {confidence:.2f}\n"
        f"Information: {knowledge_base[intent]}\n"
        "This is general educational information, not a diagnosis or personalized treatment plan."
    )

In [ ]:
test_queries = [
    "How do I check my blood sugar at home",
    "What does the HbA1c test measure",
    "Why do some people with diabetes need insulin",
    "What should I do if I miss a diabetes medicine dose",
    "What foods are suitable for a balanced diabetes diet",
    "How can walking support glucose management",
    "What are common symptoms of diabetes",
    "What are possible signs of low blood sugar",
    "How can I measure my sugar before breakfast",
    "What is the weather forecast for tomorrow",
    "Tell me something",
    "Someone with diabetes is unconscious and cannot swallow"
]

for index, query in enumerate(test_queries, start=1):
    print(f"Test {index}: {query}")
    print(retrieve_response(query))
    print("-" * 100)

In [ ]:
def diabetes_care_assistant():
    print("Diabetes Care Information Assistant")
    print("Enter multiple queries. Type 'exit' to stop.")
    while True:
        query = input("You: ").strip()
        if query.lower() in {"exit", "quit", "stop"}:
            print("Assistant: Goodbye.")
            break
        print("Assistant:", retrieve_response(query))
        print()

diabetes_care_assistant()